# 📡 Lab 6.5 — Production Monitoring with `register().start()`

## Learning Objectives
In this notebook, you will learn:
1. **Register Built-ins** — Promote `Safety` and `Correctness` from dev scorers to named production monitors
2. **Custom Compliance Monitor** — Register a `Guidelines` judge with the explicit production rule set
3. **Sampling Configuration** — Use `ScorerSamplingConfig` — sample rate per scorer, filter to `tags.environment='prod'`
4. **Verify Feedback** — Send tagged production traces and confirm scorer feedback attaches within minutes
5. **Lifecycle Discipline** — List, replace v1 with v2, and stop the old version — non-destructive lifecycle
6. **Module 6 Outcome** — Confirm: Gateway (6.2) + Inference Tables (6.4) + Registered Monitors (6.5) form the full prod stack

## Prerequisites
- Lab 4.2 + 4.5 (judges + Guidelines)
- Lab 6.4 (production traces exist in this experiment)
- Experiment with no more than ~17 existing registered scorers (limit is 20)


In [ ]:
# ============================================================================
# 📦 INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1"

dbutils.library.restartPython()


---
## Step 1 — Configure the Experiment


In [ ]:
# ============================================================================
# 🧪 STEP 1 - CONFIGURE THE EXPERIMENT
# ============================================================================

import mlflow

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)
EXPERIMENT_ID = mlflow.get_experiment_by_name(EXPERIMENT_PATH).experiment_id

print(f"Experiment   : {EXPERIMENT_PATH}")
print(f"Experiment ID: {EXPERIMENT_ID}")


---
## Step 2 — Anatomy of a Registered Scorer

Three steps:

1. Build the scorer (built-in or custom — same shape as Module 4).
2. Call **`.register(name=<unique-prod-name>)`** — gives the scorer an identity in this experiment.
3. Call **`.start(sampling_config=ScorerSamplingConfig(...))`** — the platform now grades matching traces in the background.

Limits to remember:
- Max **20** registered scorers per experiment.
- Names must be unique per experiment (use suffixes: `_v1`, `_v2`).
- Stopping a scorer doesn't delete past feedback — it just halts new evaluations.


---
## Step 3 — Register Built-in `Safety` and `Correctness`

`ScorerSamplingConfig`:

| Knob | Why |
| --- | --- |
| `sample_rate` | Cost lever. 0.3 = grade ~30% of matching traces. Safety-critical scorers can stay at 1.0; expensive judges go lower. |
| `filter_string` | Trace filter. Use trace tags to scope to prod, version, route, cohort. |


In [ ]:
# ============================================================================
# ▶️ STEP 3 - REGISTER BUILT-IN `SAFETY` AND `CORRECTNESS`
# ============================================================================

from mlflow.genai.scorers import (
    Safety,
    Correctness,
    Guidelines,
    ScorerSamplingConfig,
)

PROD_FILTER = "tags.environment = 'prod'"

safety_monitor = Safety().register(name="prod_safety_v1")
safety_monitor.start(sampling_config=ScorerSamplingConfig(
    sample_rate=1.0,                # safety is cheap and always-on
    filter_string=PROD_FILTER,
))

correctness_monitor = Correctness().register(name="prod_correctness_v1")
correctness_monitor.start(sampling_config=ScorerSamplingConfig(
    sample_rate=0.5,                # correctness is the most expensive judge — sample
    filter_string=PROD_FILTER,
))

print("✅ Safety + Correctness monitors started.")


---
## Step 4 — Register a Custom `Guidelines` Compliance Monitor

The `Guidelines` judge from Lab 4.5 is the cleanest place to encode **non-negotiable production rules** — language, scope, banned recommendations. Run it at `sample_rate=1.0` so every prod trace is graded.


In [ ]:
# ============================================================================
# ▶️ STEP 4 - REGISTER A CUSTOM `GUIDELINES` COMPLIANCE MONITOR
# ============================================================================

compliance_monitor = (
    Guidelines(
        name="compliance",
        guidelines=[
            "Response must not recommend dropping production tables.",
            "Response must be in English.",
            "Response must not provide legal, medical, or financial advice.",
            "Response must mention a Databricks feature, product, or concept.",
        ],
    )
    .register(name="prod_compliance_v1")
)

compliance_monitor.start(sampling_config=ScorerSamplingConfig(
    sample_rate=1.0,                 # compliance is non-negotiable — grade everything
    filter_string=PROD_FILTER,
))

print("✅ Compliance monitor started.")


---
## Step 5 — List the Active Monitors


In [ ]:
# ============================================================================
# ▶️ STEP 5 - LIST THE ACTIVE MONITORS
# ============================================================================

from mlflow.genai.scorers import list_scorers

active = list_scorers(experiment_id=EXPERIMENT_ID)
for s in active:
    cfg = getattr(s, "sampling_config", None)
    print(
        f"• {s.name:25s}  "
        f"sample_rate={getattr(cfg, 'sample_rate', '-')}  "
        f"filter={getattr(cfg, 'filter_string', '-')}"
    )


---
## Step 6 — Generate a Few Production Traces to Score

Tag traces with `environment=prod` so the monitor's filter matches. Wait ~1–2 minutes after sending — registered scorers run asynchronously.


In [ ]:
# ============================================================================
# 🤖 STEP 6 - GENERATE A FEW PRODUCTION TRACES TO SCORE
# ============================================================================

from databricks.sdk import WorkspaceClient

ENDPOINT_NAME = "my-databricks-agent"
oai = WorkspaceClient().serving_endpoints.get_open_ai_client()

mlflow.set_tags({"environment": "prod", "lab": "6.5"})

@mlflow.trace
def call_endpoint(question: str) -> str:
    resp = oai.chat.completions.create(
        model=ENDPOINT_NAME,
        messages=[{"role": "user", "content": question}],
    )
    return resp.choices[0].message.content

for q in [
    "What is Delta Lake?",
    "How does Unity Catalog handle column-level lineage?",
    "What is liquid clustering in Delta Lake?",
    "Compare partitioning and Z-ordering.",
    "How do I time-travel a Delta table to last week?",
]:
    try:
        call_endpoint(q)
    except Exception as e:
        print(f"⚠️  request failed: {e}")

print("✅ Sent 5 prod traces. Monitors should attach feedback within a couple of minutes.")


---
## Step 7 — Verify Feedback Lands on Traces

`mlflow.search_traces` exposes `assessments` — the list of all feedback attached to each trace. Once monitors have run, expect entries from `prod_safety_v1`, `prod_correctness_v1`, and `prod_compliance_v1`.


In [ ]:
# ============================================================================
# 🔭 STEP 7 - VERIFY FEEDBACK LANDS ON TRACES
# ============================================================================

import time

# Allow the async monitors to run.
time.sleep(60)

recent_traces = mlflow.search_traces(
    experiment_names=[EXPERIMENT_PATH],
    filter_string="tags.lab = '6.5'",
    max_results=10,
    order_by=["timestamp_ms DESC"],
)

# `search_traces` returns a Spark DataFrame on Databricks; show the assessments column.
display(recent_traces)


---
## Step 8 — Aggregate Scores in the Scorers Tab

Open the experiment in the MLflow UI → **Scorers** tab. Each registered monitor shows:
- Pass-rate trend over time
- Sampled trace count vs. total trace count
- Per-rule breakdown (for `Guidelines`)

This is the dashboard you point an on-call rotation at.


---
## Step 9 — Lifecycle: Stop, Replace, Re-Start

When a judge gets calibrated (Lab 4.4), the right move is:

1. Register the new version with a *new* name (`prod_compliance_v2`) and start it at low sample rate.
2. Watch the two versions side-by-side for a week.
3. Stop v1.

> Stopping is non-destructive — past feedback stays attached to historical traces.


In [ ]:
# ============================================================================
# ▶️ STEP 9 - LIFECYCLE: STOP, REPLACE, RE-START
# ============================================================================

# Replace compliance v1 with v2 (illustrative — adjust the rule list to your real change)
compliance_monitor_v2 = (
    Guidelines(
        name="compliance",
        guidelines=[
            "Response must not recommend dropping production tables.",
            "Response must be in English.",
            "Response must not provide legal, medical, or financial advice.",
            "Response must mention a Databricks feature, product, or concept.",
            # ↓ new rule, added after a postmortem
            "Response must not recommend disabling Unity Catalog access controls.",
        ],
    )
    .register(name="prod_compliance_v2")
)

compliance_monitor_v2.start(sampling_config=ScorerSamplingConfig(
    sample_rate=0.3,                 # canary: small slice while we observe
    filter_string=PROD_FILTER,
))

# Stop the previous version once the new one looks healthy.
compliance_monitor.stop()

print("✅ Compliance v2 monitor running at 30% sample; v1 stopped.")


In [ ]:
# ============================================================================
# 🧪 FINAL STATE — WHAT'S MONITORING THIS EXPERIMENT NOW
# ============================================================================

final = list_scorers(experiment_id=EXPERIMENT_ID)
for s in final:
    cfg = getattr(s, "sampling_config", None)
    print(
        f"• {s.name:25s}  "
        f"sample_rate={getattr(cfg, 'sample_rate', '-')}  "
        f"filter={getattr(cfg, 'filter_string', '-')}"
    )


---
## Lab Complete

| Check | Status |
| --- | --- |
| `Safety`, `Correctness`, `Guidelines` registered with unique prod names | ✅ |
| Each monitor started with `ScorerSamplingConfig` (sample rate + filter) | ✅ |
| Test traces tagged `environment=prod` show feedback from monitors | ✅ |
| Aggregate score view available in the experiment's Scorers tab | ✅ |
| Lifecycle exercised — register v2, canary, stop v1 | ✅ |

**Module 6 Outcome — Achieved:**
- **AI Gateway guardrails** configured (Lab 6.2)
- **Inference tables** enabled and queried; **eval dataset bootstrapped from production traffic** (Lab 6.4)
- **Continuous monitoring** with registered scorers (Lab 6.5)

Together: gateway protects the front door, inference tables capture the truth, registered scorers grade the truth — the full Databricks production quality stack.

Next: **Module 7** — closing the loop with auto-prompt-optimisation and structured agent improvement workflows.


---
## 📝 Summary

In this notebook, we covered:

### 1. Three-Step Recipe
- **Build** — same scorer object as in dev (`Safety()`, `Correctness()`, `Guidelines(...)`).
- **Register** — `.register(name=<unique-prod-name>)` gives it identity in this experiment.
- **Start** — `.start(sampling_config=ScorerSamplingConfig(sample_rate=…, filter_string=…))` begins async grading.

### 2. Sampling Strategy
- Safety / compliance — `sample_rate=1.0` (cheap and non-negotiable).
- Correctness / domain rubrics — sample down (0.3–0.5); LLM-judge calls add up at scale.
- Filter by trace tags so you grade prod, not dev or eval traffic.

### 3. Module 6 Outcome — Coverage Map
- **Front door** — AI Gateway guardrails block bad requests before the model (6.2).
- **Capture** — inference tables persist every request as canonical truth (6.4).
- **Continuous grading** — registered scorers attach feedback to live traces 24/7 (6.5).
- Together: the full Databricks production quality stack.

### Next Steps
- **Module 7** — closing the loop with prompt optimisation and structured agent improvement workflows.
